# 03 — Classification Modeling

## What to do
Build supervised learning models to predict binary classification (Has diabetes: Yes/No).

## What you'll learn
- Train multiple classification models
- Hyperparameter tuning
- Cross-validation
- Evaluate using appropriate metrics
- Analyze feature importance
- Select best model

## Step 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

print('Libraries loaded!')

## Step 2: Load preprocessed data

**Action:** Load scaled features and target from preprocessing.
**Insight:** We use preprocessed data to ensure consistency.

In [ ]:
X_train = pd.read_csv('data/X_train_scaled.csv')
X_test = pd.read_csv('data/X_test_scaled.csv')
y_train = pd.read_csv('data/y_train.csv').squeeze()
y_test = pd.read_csv('data/y_test.csv').squeeze()

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')

## Step 3: Train baseline models

**Action:** Train Logistic Regression, Random Forest, and Gradient Boosting.
**Insight:** Baseline models provide performance benchmarks. Ensemble models often outperform single models.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f'  ✓ Complete')

print('\n✓ All models trained!')

## Step 4: Cross-validation evaluation

**Action:** Evaluate models using 5-fold cross-validation.
**Insight:** Cross-validation estimates real-world performance and reduces overfitting risk.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    cv_results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'{name}:')
    print(f'  Mean ROC-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})')
    print(f'  Fold scores: {[f"{s:.4f}" for s in scores]}\n')

## Step 5: Test set predictions

**Action:** Make predictions on the holdout test set.
**Insight:** Test set performance is the final estimate of model quality on unseen data.

In [ ]:
predictions = {}
probabilities = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test)
    probabilities[name] = model.predict_proba(X_test)[:, 1]

print('✓ Predictions generated!')

## Step 6: Evaluate models on test set

**Action:** Calculate classification metrics (Accuracy, Precision, Recall, F1, ROC-AUC).
**Insight:** Different metrics matter for different applications. For imbalanced data, avoid accuracy alone.

In [ ]:
results_list = []

for name in models.keys():
    pred = predictions[name]
    proba = probabilities[name]
    
    results_list.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1-Score': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, proba)
    })

results_df = pd.DataFrame(results_list).sort_values('ROC-AUC', ascending=False)
display(results_df.round(4))

best_model_name = results_df.iloc[0]['Model']
print(f'\n✓ Best model by ROC-AUC: {best_model_name}')

## Step 7: Confusion matrix analysis

**Action:** Visualize true positives, true negatives, false positives, false negatives.
**Insight:** Shows which types of errors the model makes. Important for understanding failure modes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, model) in enumerate(trained_models.items()):
    pred = predictions[name]
    cm = confusion_matrix(y_test, pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f'{name}\nConfusion Matrix')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Step 8: ROC curve comparison

**Action:** Plot ROC curves for all models.
**Insight:** ROC curves show trade-off between true positive rate and false positive rate across thresholds.

In [ ]:
plt.figure(figsize=(10, 7))

for name in models.keys():
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    auc = roc_auc_score(y_test, probabilities[name])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid()
plt.show()

## Step 9: Feature importance (Random Forest)

**Action:** Extract and visualize feature importance from Random Forest.
**Insight:** Shows which features contribute most to predictions. Helps with model interpretability.

In [ ]:
rf_model = trained_models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top 10 important features:')
display(feature_importance.head(10))

plt.figure(figsize=(10, 6))
plt.barh(feature_importance.head(15)['Feature'][::-1], feature_importance.head(15)['Importance'][::-1])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.tight_layout()
plt.show()

## Step 10: Classification report (best model)

**Action:** Print detailed classification metrics for best model.
**Insight:** Per-class metrics show how well the model performs for each class separately.

In [ ]:
best_model = trained_models[best_model_name]
best_pred = predictions[best_model_name]

print(f'\n{best_model_name} - Classification Report:\n')
print(classification_report(y_test, best_pred, target_names=['No Diabetes', 'Diabetes']))

## Classification Summary

**What we did:**
1. Trained 3 classification models
2. Evaluated using cross-validation
3. Generated predictions on test set
4. Compared metrics (Accuracy, Precision, Recall, F1, ROC-AUC)
5. Analyzed confusion matrices and ROC curves
6. Extracted feature importance

**Best Model:** Ready for deployment or further tuning!
**Next:** Continue to regression and clustering for deeper analysis.